In [3]:
import os, time, pickle, sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from xgboost import XGBClassifier

In [4]:
df = pd.read_csv("/kaggle/input/datasets/deepcontractor/smoke-detection-dataset/smoke_detection_iot.csv")
print(df.shape)
print("missing values:", int(df.isnull().sum().sum()))

TARGET = "Fire Alarm"
drop_cols = [c for c in ["Unnamed: 0", "UTC", "CNT"] if c in df.columns]   #useless to the classification (remember to to remove it before uploading ...)
X = df.drop(columns=drop_cols + [TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape, "| features:", X.shape[1])

(62630, 16)
missing values: 0
Train: (50104, 12) Test: (12526, 12) | features: 12



## 1. Pipeline construction


### Task 1.1 - Logistic Regression pipeline (LR-Pipeline)

In [5]:
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
])

t0 = time.perf_counter()
lr_pipe.fit(X_train, y_train)
lr_train_time = time.perf_counter() - t0

pred = lr_pipe.predict(X_test)
lr_acc, lr_f1 = accuracy_score(y_test, pred), f1_score(y_test, pred)
print(f"LR-Pipeline  | Accuracy = {lr_acc:.4f} | F1 = {lr_f1:.4f} | train time = {lr_train_time:.3f} s")

LR-Pipeline  | Accuracy = 0.8951 | F1 = 0.9281 | train time = 0.760 s


### Task 1.2 - XGBoost pipeline (XGB-Pipeline)


In [6]:
xgb_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.1,
                          eval_metric="logloss", random_state=42, n_jobs=1)),
])

t0 = time.perf_counter()
xgb_pipe.fit(X_train, y_train)
xgb_train_time = time.perf_counter() - t0

pred = xgb_pipe.predict(X_test)
xgb_acc, xgb_f1 = accuracy_score(y_test, pred), f1_score(y_test, pred)
print(f"XGB-Pipeline | Accuracy = {xgb_acc:.4f} | F1 = {xgb_f1:.4f} | train time = {xgb_train_time:.3f} s")

XGB-Pipeline | Accuracy = 0.9999 | F1 = 0.9999 | train time = 1.345 s


In [7]:
acc_table = pd.DataFrame({
    "Accuracy": [lr_acc, xgb_acc],
    "F1-Score": [lr_f1, xgb_f1],
    "Training time (s)": [lr_train_time, xgb_train_time],
}, index=["LR-Pipeline", "XGB-Pipeline"]).round(4)
acc_table

,Accuracy,F1-Score,Training time (s)
LR-Pipeline,0.8951,0.9281,0.7596
XGB-Pipeline,0.9999,0.9999,1.3446
